In [1]:
from pathlib import Path
from collections import deque, Counter
from typing import Callable
from queue import Queue
import random
import threading
import time

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F


print("cuda available" if torch.cuda.is_available() else "cuda unavailable")
print("gpu ready" if torch.cuda.device_count() else "only cpu")

PATH_PLAYLIST = Path("../data/playlist")
PATH_CHUNKS = PATH_PLAYLIST / "mini_chunks/"   # swap to "chunks/" for production
PATH_VOCAB = PATH_PLAYLIST / "training_vocab.parquet"

cuda available
gpu ready


# Hyperparameters

In [2]:
CMIN       = 10      # min playlist appearances to keep a track
VALID_FRAC = 0.1     # fraction of chunks reserved for validation

SEED       = 0
W          = 5       # context window half-width
K          = 15      # negatives per positive
NBLOCK   = 64
EMBED_DIM  = 128
BATCH_SIZE = 16_000  # 512K+ appropriate for A100
LR         = 1e-3
NEPOCHS    = 20

PREFETCH_WORKERS = 10 # background threads for chunk preprocessing
PREFETCH_QUEUE   = 4 # max processed chunks buffered in RAM

# Vocabulary building

The production vocabulary is too large for the local environment. We replace it with a vocabulary computed on the fly from the chunks.

In [3]:
chunk_paths = sorted(PATH_CHUNKS.glob("chunk_*.parquet"))
print(f"Found {len(chunk_paths)} chunks in {PATH_CHUNKS}")

if (PROD := False):
    # Production: load pre-built vocab (fast, avoids full chunk scan)
    vocab = pd.read_parquet(PATH_VOCAB)
    if CMIN > 1:
        vocab = vocab.query("playlist_count >= @CMIN").reset_index(drop=True)
        vocab["track_id"] = np.arange(len(vocab), dtype=np.int32)
else:
    # Local / mini-chunks: build vocab from chunks on the fly
    tcounter = Counter(); pnum = 0; nint = 0
    for path in chunk_paths:
        pt = pd.read_parquet(path, columns=["playlist_rowid", "track_rowid"])
        tcounter.update(pt.groupby("track_rowid")["playlist_rowid"].nunique().to_dict())
        pnum += len(pt["playlist_rowid"].unique()); nint += len(pt)
    vocab = (
        pd.DataFrame(tcounter.items(), columns=["track_rowid", "playlist_count"])
        .query("playlist_count >= @CMIN")
        .sort_values("track_rowid")
        .reset_index(drop=True)
    )
    vocab["track_id"] = vocab.index.astype("int32")

# %% Vocab and stats
print(f"Track vocab size : {len(vocab):>10,}  (CMIN={CMIN})")
print(f"Playlist number : {pnum:>10,}")
print(f"Total interactions : {nint:>10,}")
print(f"Track count p50  : {vocab['playlist_count'].median():.0f}")
print(f"Track count p99  : {vocab['playlist_count'].quantile(0.99):.0f}")
print(f"Track count min  : {vocab['playlist_count'].min()}")
print(f"Track count max  : {vocab['playlist_count'].max()}")

Found 10 chunks in ../data/playlist/mini_chunks
Track vocab size :  1,229,514  (CMIN=10)
Playlist number :    493,579
Total interactions : 100,983,371
Track count p50  : 19
Track count p99  : 716
Track count min  : 10
Track count max  : 14278


# Memory Budget

In [4]:
vocab_size = len(vocab)
B, D = BATCH_SIZE, EMBED_DIM
MB = 1 / 1_000_000
GB = 1 / 1_000_000_000
BYTES_IN_32BIT = 4

embed    = 2 * vocab_size * D * BYTES_IN_32BIT
optim    = 2 * embed
weights_ = vocab_size * BYTES_IN_32BIT
negs     = B * K * NBLOCK * BYTES_IN_32BIT
act_fwd  = (2 + K) * B * D * BYTES_IN_32BIT
act_bwd  = act_fwd

total    = embed + optim + weights_ + act_fwd + act_bwd

print(f"Vocab size          : {vocab_size:>10,}")
print(f"Config              : EMBED_DIM={D}  BATCH_SIZE={B:,}  W={W}  K={K}  NBLOCK={NBLOCK}")
print()
print(f"Embedding tables    : {embed    * GB:>6.2f} GB  (2 tables × {vocab_size:,} × {D} × fp32)")
print(f"Optimizer state     : {optim    * GB:>6.2f} GB  (SparseAdam exp_avg + exp_avg_sq, dense after warmup)")
print(f"Weights             : {weights_ * MB:>6.1f} MB  (negative sample weights)")
print(f"Neg. sample block   : {negs     * MB:>6.1f} MB  (negative sampling reservoir)")
print(f"Activations fwd     : {act_fwd  * MB:>6.1f} MB  ({2+K} tensors × {B:,} × {D})")
print(f"Activations bwd     : {act_bwd  * MB:>6.1f} MB  (sparse grad upper bound)")
print()
print(f"Total estimate      : {total    * GB:>6.2f} GB")
if torch.cuda.is_available():
    vram_total = torch.cuda.get_device_properties(0).total_memory
    vram_free, _ = torch.cuda.mem_get_info()
    headroom = vram_free - total
    flag = "OK" if headroom > 0 else "OOM RISK"
    print()
    print(f"GPU                 : {torch.cuda.get_device_properties(0).name}")
    print(f"VRAM total          : {vram_total * GB:>6.2f} GB")
    print(f"VRAM free now       : {vram_free  * GB:>6.2f} GB")
    print(f"VRAM used now       : {(vram_total - vram_free)  * GB:>6.2f} GB")
    print(f"Headroom            : {headroom   * GB:>+6.2f} GB  [{flag}]")

Vocab size          :  1,229,514
Config              : EMBED_DIM=128  BATCH_SIZE=16,000  W=5  K=15  NBLOCK=64

Embedding tables    :   1.26 GB  (2 tables × 1,229,514 × 128 × fp32)
Optimizer state     :   2.52 GB  (SparseAdam exp_avg + exp_avg_sq, dense after warmup)
Weights             :    4.9 MB  (negative sample weights)
Neg. sample block   :   61.4 MB  (negative sampling reservoir)
Activations fwd     :  139.3 MB  (17 tensors × 16,000 × 128)
Activations bwd     :  139.3 MB  (sparse grad upper bound)

Total estimate      :   4.06 GB

GPU                 : NVIDIA GeForce RTX 2060
VRAM total          :   6.21 GB
VRAM free now       :   5.72 GB
VRAM used now       :   0.49 GB
Headroom            :  +1.66 GB  [OK]


# Train/valid split

In [5]:
rng = np.random.default_rng(SEED)
chunk_order = rng.permutation(len(chunk_paths)).tolist()
n_valid = max(1, int(len(chunk_paths) * VALID_FRAC))
valid_chunk_paths = [chunk_paths[i] for i in chunk_order[-n_valid:]]
train_chunk_paths = [chunk_paths[i] for i in chunk_order[:-n_valid]]

print(f"Train chunks : {len(train_chunk_paths)}")
print(f"Valid chunks : {len(valid_chunk_paths)}")

Train chunks : 9
Valid chunks : 1


# Precomputing subsample probabilities and weights

In [6]:
counts = vocab["playlist_count"].values.astype(np.float64)
freq = counts / counts.sum()

sub_t = float(np.quantile(freq, 0.99))
KEEP_PROB = np.minimum(1.0, np.sqrt(sub_t / freq)).astype(np.float32)  # indexed by track_id

w75 = counts ** 0.75
NEG_WEIGHTS = torch.tensor(w75 / w75.sum(), dtype=torch.float32)

del counts, freq, w75, sub_t


def subsample(pt: pd.DataFrame, rng: np.random.Generator) -> pd.DataFrame:
    p = KEEP_PROB[pt["track_id"].values]
    mask = rng.random(len(pt)) < p
    return pt[mask].reset_index(drop=True)

# OOV filtering

In [7]:
VOCAB_ROWIDS = vocab["track_rowid"].values   # sorted int64
VOCAB_TIDS   = vocab["track_id"].values      # corresponding track_id


def remap_chunk(pt: pd.DataFrame) -> pd.DataFrame:
    """Filter OOV tracks and add track_id via binary search (replaces pd.merge)."""
    rowids = pt["track_rowid"].values
    idx = np.searchsorted(VOCAB_ROWIDS, rowids)
    idx = np.clip(idx, 0, len(VOCAB_ROWIDS) - 1)
    match = VOCAB_ROWIDS[idx] == rowids
    out = pt[match].copy()
    out["track_id"] = VOCAB_TIDS[idx[match]]
    return out

# Compute center-context pairs

In [8]:
def precompute_pairs(pt: pd.DataFrame, W: int) -> torch.Tensor:
    """Generate all (center, context) skip-gram pairs for a chunk.

    Works entirely in numpy: finds playlist boundaries, builds per-track
    metadata with np.repeat, then loops over the 2W offsets (~10 iters)
    doing vectorised index arithmetic across all tracks at once.
    """
    # sort by playlist_rowid so boundary detection works even if the
    # parquet rows aren't perfectly grouped (no ORDER BY in the SQL)
    if not pt["playlist_rowid"].is_monotonic_increasing:
        pt = pt.sort_values("playlist_rowid", kind="mergesort")

    pids = pt["playlist_rowid"].values
    tids = pt["track_id"].values

    # --- playlist boundaries ---
    breaks = np.empty(len(pids), dtype=bool)
    breaks[0] = True
    breaks[1:] = pids[1:] != pids[:-1]
    starts = np.flatnonzero(breaks)
    lengths = np.diff(np.append(starts, len(pids)))

    # keep only playlists with >= 2 tracks
    valid = lengths >= 2
    starts, lengths = starts[valid], lengths[valid]
    if len(starts) == 0:
        return torch.empty(2, 0, dtype=torch.long)

    # --- per-track metadata (vectorised via np.repeat) ---
    total = lengths.sum()
    pos = np.arange(total, dtype=np.int32)
    cum = np.cumsum(lengths)
    # position within playlist: 0,1,..,L-1, 0,1,..,L-1, ...
    pos -= np.repeat(np.concatenate(([0], cum[:-1])), lengths).astype(np.int32)

    flat_start = np.repeat(starts, lengths)    # absolute start index per track
    flat_len   = np.repeat(lengths, lengths)   # playlist length per track

    # effective window per track: min(2W, L-1), mirrors the original code
    ew = np.minimum(2 * W, flat_len - 1)

    # --- generate pairs for each offset ---
    all_centers = []
    all_contexts = []
    for k in range(-W, W + 1):
        if k == 0:
            continue
        # offset k is valid iff it falls inside range((-ew)//2, ew//2+1),
        # matching the original playlist_pairs logic exactly
        mask = ((-ew) // 2 <= k) & (k <= ew // 2)
        m_pos   = pos[mask]
        m_start = flat_start[mask]
        m_len   = flat_len[mask]

        ctx_pos = (m_pos + k) % m_len
        ctx_idx = m_start + ctx_pos

        all_centers.append(tids[m_start + m_pos])
        all_contexts.append(tids[ctx_idx])

    if not all_centers:
        return torch.empty(2, 0, dtype=torch.long)

    centers  = np.concatenate(all_centers)
    contexts = np.concatenate(all_contexts)
    return torch.from_numpy(np.stack([centers, contexts]))

# Chunk processing

In [9]:
def process_chunk(path: Path, chunk_rng: np.random.Generator) -> torch.Tensor:
    pt = pd.read_parquet(path, columns=["playlist_rowid", "track_rowid"])
    pt = remap_chunk(pt)
    pt = subsample(pt, chunk_rng)
    pairs = precompute_pairs(pt, W)
    if pairs.shape[1] == 0:
        return pairs
    perm = torch.randperm(pairs.shape[1])
    return pairs[:, perm].pin_memory()

In [10]:
class PrefetchPairStream:
    """Producer-consumer pair stream with background chunk preprocessing.

    A pool of daemon threads reads and preprocesses chunks into a bounded queue.
    The training loop calls next_batch() which splices across chunk boundaries.
    Each chunk gets its own deterministic RNG seeded from (seed, epoch, chunk_index).
    """

    def __init__(
        self,
        chunk_paths: list[Path],
        epoch: int,
        seed: int,
        n_workers: int = PREFETCH_WORKERS,
        queue_size: int = PREFETCH_QUEUE,
    ):
        self._queue: Queue = Queue(maxsize=queue_size)
        self._buffer = torch.empty(2, 0, dtype=torch.long)

        # Each (index, path) pair is grabbed by a worker under lock.
        self._work = deque(enumerate(chunk_paths))
        self._lock = threading.Lock()
        self._epoch = epoch
        self._seed = seed
        self._n_workers = n_workers
        self._sentinels_remaining = n_workers

        self._threads = [
            threading.Thread(target=self._worker, daemon=True) for _ in range(n_workers)
        ]
        for t in self._threads:
            t.start()

    def _worker(self):
        while True:
            with self._lock:
                if not self._work:
                    break
                idx, path = self._work.popleft()
            chunk_rng = np.random.default_rng(self._seed + self._epoch * 10_000 + idx)
            tensor = process_chunk(path, chunk_rng)
            self._queue.put(tensor)
        self._queue.put(None)  # sentinel — one per worker

    def next_batch(self, batch_size: int) -> torch.Tensor:
        while self._buffer.shape[1] < batch_size:
            if self._sentinels_remaining == 0:
                # all workers done, return whatever is left
                remainder = self._buffer
                self._buffer = torch.empty(2, 0, dtype=torch.long)
                return remainder
            item = self._queue.get()
            if item is None:
                self._sentinels_remaining -= 1
                continue
            if item.shape[1] == 0:
                continue
            self._buffer = torch.cat([self._buffer, item], dim=1) if self._buffer.shape[1] > 0 else item

        batch = self._buffer[:, :batch_size]
        self._buffer = self._buffer[:, batch_size:]
        return batch


def get_pair_stream_serial(chunk_paths: list[Path], epoch: int, seed: int) -> Callable:
    """Single-threaded fallback (e.g. for validation with few chunks)."""
    buffer = torch.empty(2, 0, dtype=torch.long)
    paths = deque(enumerate(chunk_paths))

    def next_batch(batch_size: int) -> torch.Tensor:
        nonlocal buffer
        while buffer.shape[1] < batch_size:
            if not paths:
                remainder = buffer
                buffer = torch.empty(2, 0, dtype=torch.long)
                return remainder
            idx, path = paths.popleft()
            chunk_rng = np.random.default_rng(seed + epoch * 10_000 + idx)
            new = process_chunk(path, chunk_rng)
            if new.shape[1] == 0:
                continue
            buffer = torch.cat([buffer, new], dim=1) if buffer.shape[1] > 0 else new

        batch = buffer[:, :batch_size]
        buffer = buffer[:, batch_size:]
        return batch

    return next_batch

# Model definition

In [11]:
class Word2Vec(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int):
        super().__init__()
        self.embeddings_in = nn.Embedding(
            num_embeddings=vocab_size, embedding_dim=embed_dim, sparse=True
        )
        self.embeddings_out = nn.Embedding(
            num_embeddings=vocab_size, embedding_dim=embed_dim, sparse=True
        )
        nn.init.uniform_(self.embeddings_in.weight,  -0.5 / embed_dim, 0.5 / embed_dim)
        nn.init.uniform_(self.embeddings_out.weight, -0.5 / embed_dim, 0.5 / embed_dim)

    def forward(
        self, center: torch.Tensor, context: torch.Tensor, negatives: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        ecenter   = self.embeddings_in(center)
        econtext  = self.embeddings_out(context)
        enegative = self.embeddings_out(negatives)

        pos_score = (ecenter * econtext).sum(dim=1)
        neg_score = torch.bmm(enegative, ecenter.unsqueeze(2)).squeeze(2)

        return pos_score, neg_score

    @property
    def track_embeddings(self) -> torch.Tensor:
        return self.embeddings_in.weight.detach()


def skipgram_loss(pos_score: torch.Tensor, neg_score: torch.Tensor) -> torch.Tensor:
    pos_loss = F.logsigmoid(pos_score)
    neg_loss = F.logsigmoid(-neg_score).sum(dim=1)
    return -(pos_loss + neg_loss).mean()

# Training

In [12]:
def get_nsampler(weights_gpu: torch.Tensor, K: int, batch_size: int, block: int = 64) -> Callable:
    total = batch_size * K * block
    cache = [torch.empty(0, dtype=torch.long, device=weights_gpu.device), 0]

    def sample(n: int) -> torch.Tensor:
        needed = n * K
        if cache[1] + needed > cache[0].shape[0]:
            cache[0] = torch.multinomial(weights_gpu, total, replacement=True)
            cache[1] = 0
        start = cache[1]
        cache[1] = start + needed
        return cache[0][start : start + needed].view(n, K)

    return sample

In [ ]:
from torch.optim import SparseAdam
from torch.optim.lr_scheduler import ReduceLROnPlateau

device = torch.device("cuda")

weights_gpu = NEG_WEIGHTS.to(device)
neg_sample  = get_nsampler(weights_gpu, K, BATCH_SIZE, NBLOCK)

torch.manual_seed(SEED)
model     = Word2Vec(vocab_size=len(vocab), embed_dim=EMBED_DIM).to(device)
optimizer = SparseAdam(model.parameters(), lr=LR)
scheduler = ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

history    = {"train": [], "valid": [], "lr": []}
best_valid = float("inf")
best_state = None
best_epoch = -1

w = len(str(NEPOCHS))

for epoch in range(NEPOCHS):
    t0 = time.perf_counter()

    torch.manual_seed(SEED + epoch)

    # --- train ---
    random.shuffle(train_chunk_paths)
    stream = PrefetchPairStream(train_chunk_paths, epoch=epoch, seed=SEED)

    model.train()
    steps, epoch_loss = 0, 0.0
    while True:
        batch = stream.next_batch(BATCH_SIZE)
        if batch.shape[1] == 0:
            break
        c = batch[0].to(device, non_blocking=True)
        x = batch[1].to(device, non_blocking=True)
        n = neg_sample(len(c))
        optimizer.zero_grad()
        loss = skipgram_loss(*model(c, x, n))
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        steps += 1
    t1 = time.perf_counter()

    # --- valid (serial — few chunks, not worth threading) ---
    valid_stream = get_pair_stream_serial(valid_chunk_paths, epoch=epoch, seed=SEED)

    model.eval()
    vsteps, vloss = 0, 0.0
    with torch.no_grad():
        while True:
            batch = valid_stream(BATCH_SIZE)
            if batch.shape[1] == 0:
                break
            c = batch[0].to(device, non_blocking=True)
            x = batch[1].to(device, non_blocking=True)
            n = neg_sample(len(c))
            vloss  += skipgram_loss(*model(c, x, n)).item()
            vsteps += 1

    t2 = time.perf_counter()

    train_loss = epoch_loss / max(steps,  1)
    valid_loss = vloss      / max(vsteps, 1)
    lr = optimizer.param_groups[0]["lr"]
    history["train"].append(train_loss)
    history["valid"].append(valid_loss)
    history["lr"].append(lr)

    is_best = valid_loss < best_valid
    if is_best:
        best_valid = valid_loss
        best_epoch = epoch + 1
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    scheduler.step(valid_loss)

    print(
        f"epoch {epoch+1:{w}}/{NEPOCHS}"
        f"  │  train {train_loss:.4f} ({steps} batches)"
        f"  valid {valid_loss:.4f} ({vsteps} batches)"
        f"  │  lr {lr:.2e}"
        f"  │  {t2-t0:.0f}s  (train {t1-t0:.0f}s  valid {t2-t1:.0f}s)"
        + ("  *" if is_best else "")
    )

model.load_state_dict(best_state)
print(f"\nRestored best model from epoch {best_epoch}  (valid {best_valid:.4f})")

epoch  1/20  │  train 2.6661 (34558 batches)  valid 2.1486 (6232 batches)  │  lr 1.00e-03  │  814s  (train 792s  valid 21s)  *
epoch  2/20  │  train 1.6649 (34558 batches)  valid 1.9345 (6232 batches)  │  lr 1.00e-03  │  1654s  (train 1633s  valid 21s)  *


### History

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(4, 4))
epochs_range = range(1, len(history["train"]) + 1)
ax.plot(epochs_range, history["train"], label="train")
ax.plot(epochs_range, history["valid"], label="valid")
ax.set_xlabel("epoch")
ax.set_ylabel("loss")
ax.legend()
plt.show()

# Testing

In [ ]:
lookup = pd.read_parquet(100983371
    columns=["track_rowid", "track_name", "artist_name", "track_popularity"],
)
lookup = lookup.merge(vocab, on="track_rowid", how="inner")
print(f"{len(lookup):,} tracks in lookup (vocab size {len(vocab):,})")
lookup.head()

# L2-normalise once — cosine sim becomes a matmul
emb      = model.track_embeddings.to(device)
emb_norm = emb / emb.norm(dim=1, keepdim=True)


def find_neighbours(query: str, k: int = 10, diverse: bool = True) -> pd.DataFrame:
    matches = lookup[lookup["track_name"].str.contains(query, case=False, na=False)]
    if matches.empty:
        print(f"No tracks matching '{query}'")
        return pd.DataFrame()

    row = matches.sort_values("track_popularity", ascending=False).iloc[0]
    tid = row["track_id"]
    print(f"Query: {row['track_name']} — {row['artist_name']}  (pop {row['track_popularity']})")

    sims = (emb_norm[tid] @ emb_norm.T).cpu().numpy()

    candidates = lookup.copy()
    candidates["similarity"] = sims[candidates["track_id"].values]
    candidates = candidates[candidates["track_id"] != tid]
    candidates = candidates.sort_values("similarity", ascending=False)
    if diverse:
        candidates = candidates.drop_duplicates(subset="artist_name")
    return candidates.head(k)[["track_name", "artist_name", "track_popularity", "similarity"]]

In [ ]:
queries = [
    "Holland, 1945",
    "when you sleep",
    "Interzone",
    "About A Girl",
    "Battery",
    "Xtal",
    "Altibzz",
    "N.Y. State of Mind",
    "Mathematics",
    "Nuthin' But A 'G' Thang",
    "Figaro",
    "Bela Lugosi's Dead",
    "Just Like Heaven",
    "Once in a Lifetime",
    "Jesus 'Tod",
    "Valvonauta",
]

for q in queries:
    display(find_neighbours(q))
    print()
